# Clearvale: WM operations, item-level template RSA post hoc

Study-specific statistics for Clearvale's working-memory task, run on the core pipeline's group raw tables. The core pipeline (`examples/clearvale/configs/config-cv-kim2020-denoised.json`, the Kim et al. 2020 method) builds feature-weighted item templates from the localizer. It then scores every WM view event against them, as a 30-TR (13.8 s) timecourse (`wm_timecourse`) and as encoding fidelity (`encoding_fidelity`). The analyses follow `_interactive_notebooks/kim2020_removal_operations.ipynb`, adapted to Clearvale.

**Design.** Each trial shows a face or place image (2.77 s, 0.00–2.77 s), then an operation cue (2.77 s, about 2.75–5.52 s): **maintain**, **suppress**, **switch** (think of an object instead; the objects are the localizer's neutral items) or **clear**. A fixation of 3.3–4.5 s follows, and the next trial starts about 9 s after onset. Valence comes from the run: WMneg runs show negative images, WMpos runs positive ones.

1. **Timecourses** of target similarity per operation.
2. **Removal**: each operation minus maintain, baseline-corrected, tested in 3-TR bins from the operation cue. The removal onset is the first bin reliably below maintain.
3. **Operation × valence**: removal separately for negative and positive images, and whether valence changes it.
4. **Switch-target tracking**: the switched-to object, if the core run scored it (see that section).
5. **Encoding fidelity**: target > same subcategory > same category > others, overall and by valence.
6. **Proactive interference**: encoding fidelity of trial N+1 when it repeats trial N's category, by the operation on trial N.

Each section draws its figures first, then its statistics.

In [ ]:
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

# run from the repository root, so the relative paths in the configs resolve as in the pipeline
REPO = os.getcwd()
while not os.path.isfile(os.path.join(REPO, "workflows", "run_pipeline.py")) and os.path.dirname(REPO) != REPO:
    REPO = os.path.dirname(REPO)
os.chdir(REPO)
sys.path.insert(0, REPO)

import _interactive_notebooks.rsa_posthoc_common as common
from _interactive_notebooks.rsa_posthoc_common import (
    add_previous_trial_columns, baseline_correct, bin_means, compare_groups_by_bin, complete_trials, difference_scores,
    equal_trial_bootstrap, fdr, rm_tukey,
    first_significant_bin, load_config, load_scores, make_bins, one_sample_by_bin, paired_contrast, pairwise_tests,
    plot_points, plot_timecourse, rm_anova, save_outputs, subject_means, subtract_reference)

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)

## Configuration -- edit these

In [ ]:
# -- which core run --
CONFIG_PATH = os.environ.get("RSA_CONFIG", "examples/clearvale/configs/config-cv-kim2020-denoised.json")
# the folder that holds group/ (e.g. the HPC group results unzipped here); None: the config's pipeline.output_dir
RESULTS_DIR = os.environ.get("RSA_RESULTS_DIR", "data/cv-bids/out/hpc_kim2020")
# inclusion list: None (every subject in the tables), a list ["001", "004"], or a text file with one subject per line
SUBJECTS = os.environ.get("RSA_SUBJECTS") or None
ROI = "vvs"
LEVEL = "item"                        # feature-weighting level
TIMECOURSE_SET = "wm_timecourse"      # target-set names in template_rsa.targets
FIDELITY_SET = "encoding_fidelity"

# -- design columns (events.tsv from make_events.py) --
OPERATION_COL = "operation"
REFERENCE = "maintain"                # removal = operation minus maintain
OPERATIONS = ["maintain", "suppress", "switch", "clear"]
COLORS = {"maintain": "green", "suppress": "red", "switch": "blue", "clear": "orange"}  # as in the core report
VALENCE_COL = "valence"               # negative (WMneg runs) / positive (WMpos runs)
VALENCES = ["negative", "positive"]
CATEGORY_COL = "category"             # face / place
SWITCH_SETS = ["switch_category", "switch_item"]  # extra-set names for the switched-to object, if the core scored them

# -- timing, in seconds (converted with the data's TR) --
BASELINE_SECONDS = (0.0, 2.76)        # subtracted from each curve: the view period (TRs 0-5 at 0.46 s)
STAT_WINDOW_SECONDS = (2.76, 9.66)    # tested: operation cue onward, as in Kim 2020 (TRs 6-20)
BIN_SECONDS = 1.38                    # bin width (3 TRs)

# -- method choices: "default" (this pipeline) or "spm12_aligned" (the lab's cval_mni_blanca_pipeline_spm12;
#    see examples/clearvale/pipeline_alignment.md); the preset only fills the settings below
PRESET = os.environ.get("RSA_POSTHOC_PRESET", "default")
BASELINE_MODE = "subject"             # "subject": each subject's own view-period mean; "group": Kim's MATLAB code
                                      # (one group mean per operation x valence); "none": raw similarity
COMPLETE_TRIALS_ONLY = False          # True: only trials scored at every TR of the timecourse window
VALENCE_CONTRAST = ("negative", "positive")   # valence differences are a - b
NPLUS1_PLANNED_PAIRS = [("clear", "suppress"), ("maintain", "suppress"), ("switch", "suppress")]  # Kim Fig. 5b
NPLUS1_BOOTSTRAP_TRIALS = 15          # Kim's equal-trial bootstrap (trials per subject x cell); None: skip
if PRESET == "spm12_aligned":
    BASELINE_MODE, COMPLETE_TRIALS_ONLY, VALENCE_CONTRAST = "none", True, ("positive", "negative")
elif PRESET != "default":
    raise ValueError(f"PRESET must be 'default' or 'spm12_aligned' (got {PRESET!r})")

# -- statistics --
ALPHA = 0.05
FDR_METHOD = "fdr_bh"
N_BOOTSTRAP = 1000
SEED = 0

OUTPUT_DIR = os.environ.get("RSA_POSTHOC_OUT")  # None: <RESULTS_DIR>/posthoc/clearvale[_<preset>]/roi-<ROI>_level-<LEVEL>
common.configure(globals())

## Load the group raw tables

One row per subject × trial × TR × comparison set; every events.tsv column rides along. With an inclusion list, the other subjects are dropped here.

In [ ]:
cfg = load_config(CONFIG_PATH)
if RESULTS_DIR:
    cfg["pipeline"]["output_dir"] = RESULTS_DIR
OUTPUT_DIR = OUTPUT_DIR or os.path.join(cfg["pipeline"]["output_dir"], "posthoc",
                                        "clearvale" + ("" if PRESET == "default" else f"_{PRESET}"), f"roi-{ROI}_level-{LEVEL}")

def read_subjects(spec):
    if not spec:
        return None
    if isinstance(spec, str) and os.path.isfile(spec):
        lines = [l.split("#")[0].strip() for l in open(spec)]
        spec = [s for l in lines for s in l.replace(",", " ").split()]
    elif isinstance(spec, str):
        spec = spec.replace(",", " ").split()
    return sorted({str(s).removeprefix("sub-") for s in spec})

INCLUDED = read_subjects(SUBJECTS)

def load(target_set):
    df = load_scores(cfg, ROI, target_set, LEVEL)
    if INCLUDED:
        missing = sorted(set(INCLUDED) - set(df["subject"]))
        if missing:
            print(f"  (!) {target_set}: no scores for included subject(s) {missing}")
        df = df[df["subject"].isin(INCLUDED)]
    return df[df[OPERATION_COL].isin(OPERATIONS)].reset_index(drop=True)

tc = load(TIMECOURSE_SET)
ef_all = load(FIDELITY_SET)
N_TC_TRS = int(tc["tr_offset"].max()) + 1
if COMPLETE_TRIALS_ONLY:
    before = tc.drop_duplicates(["subject", "run_id", "event_index"]).shape[0]
    tc = complete_trials(tc[tc["comparison"] == "target"], N_TC_TRS).pipe(
        lambda kept: tc.merge(kept[["subject", "run_id", "event_index"]].drop_duplicates()))
    print(f"complete trials only: {tc.drop_duplicates(['subject', 'run_id', 'event_index']).shape[0]} of {before} trials "
          f"have all {N_TC_TRS} TRs")
BASELINE_LABEL = {"subject": "baseline-corrected", "group": "group-baseline-corrected (Kim)", "none": "raw"}[BASELINE_MODE]

TR = float(np.median(np.diff(np.sort(tc["time_s"].unique()))))
to_tr = lambda seconds: int(round(seconds / TR))
BASELINE_TRS = (to_tr(BASELINE_SECONDS[0]), to_tr(BASELINE_SECONDS[1]))
FIRST_TR, LAST_TR = to_tr(STAT_WINDOW_SECONDS[0]), to_tr(STAT_WINDOW_SECONDS[1]) - 1
BINS = make_bins(FIRST_TR, LAST_TR, max(1, to_tr(BIN_SECONDS)))
STAT_SPAN = (FIRST_TR * TR, (LAST_TR + 1) * TR)
OPS = [o for o in OPERATIONS if o in set(tc[OPERATION_COL])]
REMOVAL_OPS = [o for o in OPS if o != REFERENCE]

trials = tc.drop_duplicates(["subject", "run_id", "event_index"])
print(f"{tc['subject'].nunique()} subjects, {len(trials)} trials, TR = {TR:g} s, {tc['tr_offset'].max() + 1} TRs per trial")
print(f"baseline TRs {BASELINE_TRS[0]}-{BASELINE_TRS[1] - 1}; tested TRs {FIRST_TR}-{LAST_TR} in bins {BINS}")
print("comparison sets:", sorted(tc["comparison"].unique()), "|", sorted(ef_all["comparison"].unique()))
display(trials.groupby([VALENCE_COL, OPERATION_COL]).size().unstack()[OPS])
# comparison sets in the encoding-fidelity table: target, related_subcategory (same subcategory = valence x
# category), related_category (same category), others (other categories); older runs have one "related" set
SETS = [c for c in ("target", "related_subcategory", "related_category", "related", "others") if c in set(ef_all["comparison"])]
SET_LABELS = {"related_subcategory": "same subcategory", "related_category": "same category"}
print("encoding-fidelity sets:", SETS)
figures, tables = {}, {}

## 1. Timecourses by operation

Trials are averaged within subject first; bands are ± 1 SEM across subjects. The grey span is the tested window (operation cue onward); the dashed lines mark the operation cue and the median next-trial onset.

In [ ]:
sub = subject_means(tc, [OPERATION_COL, "comparison", "tr_offset", "time_s"])
comparisons = [c for c in ("target", "related_subcategory", "related_category", "related", "others") if c in set(sub["comparison"])]
next_onset = trials.sort_values(["subject", "run_id", "onset"]).groupby(["subject", "run_id"])["onset"].diff().median()

fig, axes = plt.subplots(1, len(comparisons), figsize=(5.5 * len(comparisons), 4), sharey=True, squeeze=False)
for ax, comp in zip(axes.flat, comparisons):
    plot_timecourse(ax, sub[sub["comparison"] == comp], OPERATION_COL, f"{comp} similarity (raw)", colors=COLORS, shade=STAT_SPAN)
    for x, label in ((BASELINE_SECONDS[1], "operation cue"), (next_onset, "next trial")):
        ax.axvline(x, color="0.4", lw=0.8, ls="--")
        ax.text(x, 0.98, f" {label}", transform=ax.get_xaxis_transform(), fontsize=7, va="top", color="0.3")
fig.suptitle(f"{ROI}, {LEVEL}-weighted templates: similarity from view onset", fontsize=11)
fig.tight_layout()
figures["1_raw_timecourses"] = fig
plt.show()

## 2. Removal: each operation minus maintain

Each subject × operation curve is baseline-corrected as set by `BASELINE_MODE` (default: its own mean over the view period; `group`: one group mean per operation, as in Kim's MATLAB code; `none`: raw, the other lab pipeline's primary result), then maintain's curve is subtracted at every TR, so **y = 0 is maintain**. Below 0 means less of the item than when maintaining it.

Per bin, each operation gets a one-sample t-test against 0. The removal onset is the first bin reliably below 0. `p_fdr` corrects across the bins of each operation. Each bin also gets a repeated-measures ANOVA across the three removal operations, with FDR-corrected pairwise tests.

In [ ]:
bc = baseline_correct(sub, BASELINE_TRS, [OPERATION_COL, "comparison"], mode=BASELINE_MODE)
removal = subtract_reference(bc[bc["comparison"] == "target"], OPERATION_COL, REFERENCE, ["tr_offset", "time_s"])
removal_binned = bin_means(removal, BINS, [OPERATION_COL])
removal_tests = one_sample_by_bin(removal_binned, [OPERATION_COL])
removal_onsets = first_significant_bin(removal_tests, [OPERATION_COL], direction="below", p_col="p")
between_operations = compare_groups_by_bin(removal_binned, OPERATION_COL)

fig, ax = plt.subplots(figsize=(7.5, 4.3))
plot_timecourse(ax, removal, OPERATION_COL, f"target similarity, operation minus {REFERENCE} ({BASELINE_LABEL})",
                tests=removal_tests, tr_seconds=TR, p_col="p_fdr", colors=COLORS, shade=STAT_SPAN)
fig.tight_layout()
figures["2_removal_minus_maintain"] = fig
plt.show()

print("Removal onset (first bin below maintain, uncorrected p < ALPHA); bars under the curves mark p_fdr < ALPHA:")
display(removal_onsets[[OPERATION_COL, "bin", "mean", "t", "p", "p_fdr"]])
display(removal_tests)
display(between_operations)
tables.update(removal_vs_maintain_by_bin=removal_tests, removal_onsets=removal_onsets,
              removal_between_operations=between_operations)

## 3. Operation × valence

The same removal curves, computed separately for negative (WMneg) and positive (WMpos) trials. Baseline and maintain are each taken within valence.

- **Per bin:** a paired t-test of negative minus positive removal, per operation (`p_fdr` across the bins of each operation).
- **Whole window:** the mean removal over the tested window, in a 3 (operation) × 2 (valence) repeated-measures ANOVA.

In [ ]:
from statsmodels.stats.anova import AnovaRM

sub_v = subject_means(tc[tc["comparison"] == "target"], [VALENCE_COL, OPERATION_COL, "tr_offset", "time_s"])
bc_v = baseline_correct(sub_v, BASELINE_TRS, [VALENCE_COL, OPERATION_COL], mode=BASELINE_MODE)
removal_v = subtract_reference(bc_v, OPERATION_COL, REFERENCE, [VALENCE_COL, "tr_offset", "time_s"])
removal_v_binned = bin_means(removal_v, BINS, [VALENCE_COL, OPERATION_COL])
removal_v_tests = one_sample_by_bin(removal_v_binned, [VALENCE_COL, OPERATION_COL])
valence_by_bin = paired_contrast(removal_v_binned, VALENCE_COL, *VALENCE_CONTRAST, by=[OPERATION_COL, "bin_start", "bin"])
valence_by_bin["p_fdr"] = valence_by_bin.groupby(OPERATION_COL)["p"].transform(lambda p: fdr(p))

fig, axes = plt.subplots(1, len(VALENCES), figsize=(12, 4.3), sharey=True)
for ax, val in zip(axes, VALENCES):
    plot_timecourse(ax, removal_v[removal_v[VALENCE_COL] == val], OPERATION_COL, f"{val}: operation minus {REFERENCE}",
                    tests=removal_v_tests[removal_v_tests[VALENCE_COL] == val], tr_seconds=TR, p_col="p_fdr",
                    colors=COLORS, shade=STAT_SPAN)
fig.tight_layout()
figures["3a_removal_by_valence"] = fig
plt.show()

# whole-window removal: subject x valence x operation
window = removal_v[(removal_v["tr_offset"] >= FIRST_TR) & (removal_v["tr_offset"] <= LAST_TR)]
window = window.groupby(["subject", VALENCE_COL, OPERATION_COL], as_index=False)["z"].mean()
complete = window.groupby("subject").size()
window_c = window[window["subject"].isin(complete[complete == len(VALENCES) * len(REMOVAL_OPS)].index)]
aov = AnovaRM(window_c, "z", "subject", within=[OPERATION_COL, VALENCE_COL]).fit().anova_table.reset_index()
aov = aov.rename(columns={"index": "effect", "F Value": "F", "Num DF": "df1", "Den DF": "df2", "Pr > F": "p"})
aov["n"] = window_c["subject"].nunique()
valence_window = paired_contrast(window, VALENCE_COL, *VALENCE_CONTRAST, by=[OPERATION_COL])

fig, ax = plt.subplots(figsize=(6, 4))
plot_points(ax, window, OPERATION_COL, VALENCE_COL, f"mean removal over {STAT_SPAN[0]:.2f}-{STAT_SPAN[1]:.2f} s",
            order=REMOVAL_OPS, hue_order=VALENCES)
ax.set_ylabel(f"operation minus {REFERENCE} (Fisher z)", fontsize=9)
fig.tight_layout()
figures["3b_removal_window_by_valence"] = fig
plt.show()

print("Operation x valence repeated-measures ANOVA on whole-window removal:")
display(aov)
print(f"{VALENCE_CONTRAST[0]} minus {VALENCE_CONTRAST[1]}, whole window:")
display(valence_window)
print(f"{VALENCE_CONTRAST[0]} minus {VALENCE_CONTRAST[1]}, per bin:")
display(valence_by_bin)
tables.update(removal_by_valence_by_bin=removal_v_tests, valence_difference_by_bin=valence_by_bin,
              removal_window_operation_x_valence_anova=aov, valence_difference_window=valence_window)

## 4. Switch-target tracking

On switch trials, participants replace the image with an object: one of the localizer's neutral items, named in `switch_target`, `switch_category` = `item` and `switch_item` = its localizer image. In Kim 2020's replace analysis, the replaced image should fall while the switched-to category or item rises.

The core run scores this only if the timecourse target set has extra sets for it. Add this to `template_rsa.targets.wm_timecourse` in the config and rerun stages 2 and 3:

```json
"extra_sets": {"switch_category": {"column": "switch_category", "level": "category"},
               "switch_item":     {"column": "switch_item",     "level": "item"}}
```

Events made before the relabel call the item column `switch_exemplar`. Early-list sessions (sub-001–029) used 5 words that have no localizer image, so they get a `switch_category` score but no `switch_item` score (see `analysis_deviations.md`).

In [ ]:
present = [s for s in SWITCH_SETS if s in set(bc["comparison"])]
if not present:
    print(f"no {SWITCH_SETS} scores in {TIMECOURSE_SET} -- add the extra_sets above to the config and rerun the core stages")
else:
    sw = bc[(bc[OPERATION_COL] == "switch") & bc["comparison"].isin(["target", *present])]
    sw_binned = bin_means(sw, BINS, ["comparison"])
    sw_tests = one_sample_by_bin(sw_binned, ["comparison"])
    sw_onsets = first_significant_bin(sw_tests[sw_tests["comparison"].isin(present)], ["comparison"], direction="above", p_col="p")
    crossover = pd.concat([paired_contrast(sw_binned, "comparison", s, "target", by=["bin_start", "bin"]) for s in present])

    fig, ax = plt.subplots(figsize=(7.5, 4.3))
    plot_timecourse(ax, sw, "comparison", f"switch trials: replaced image vs. switched-to object ({BASELINE_LABEL})",
                    tests=sw_tests, tr_seconds=TR, p_col="p_fdr", shade=STAT_SPAN)
    fig.tight_layout()
    figures["4_switch_tracking"] = fig
    plt.show()
    print("Switch-target onset (first bin above baseline):")
    display(sw_onsets[["comparison", "bin", "mean", "t", "p"]])
    display(crossover)
    tables.update(switch_by_bin=sw_tests, switch_onsets=sw_onsets, switch_minus_target=crossover)

## 5. Encoding fidelity

The similarity of each view's pattern (window mean from onset + 4.6 s to offset + 4.6 s) to the presented image's template (`target`), to the other images of its subcategory (`related_subcategory`: same valence and category, the closest other images), to the other images of its category (`related_category`: both valences, the broader set) and to the images of the other categories (`others`). This is a one-way repeated-measures ANOVA with FDR-corrected paired t-tests.

The view comes before the operation cue, so operation should not matter here, and a check across operations is included. The valence panel shows whether negative and positive images are encoded equally faithfully.

In [ ]:
ef_sub = subject_means(ef_all, ["comparison"])
F, p, n = rm_anova(ef_sub, "z", "comparison")
fidelity_pairs = pairwise_tests(ef_sub, "z", "comparison", levels=SETS)
fidelity_anova = [{"test": "rm_anova across " + " / ".join(SETS), "F": F, "p": p, "n": n}]

ef_target = ef_all[ef_all["comparison"] == "target"]
ef_op = subject_means(ef_target, [OPERATION_COL])
F, p, n = rm_anova(ef_op, "z", OPERATION_COL)
fidelity_anova.append({"test": "rm_anova of target fidelity across operations (expected null)", "F": F, "p": p, "n": n})
ef_val = subject_means(ef_all, [VALENCE_COL, "comparison"])
ef_val_target = ef_val[ef_val["comparison"] == "target"]
fidelity_valence = paired_contrast(ef_val_target, VALENCE_COL, *VALENCE_CONTRAST)
fidelity_anova = pd.DataFrame(fidelity_anova)

fig, axes = plt.subplots(1, 3, figsize=(14, 4), gridspec_kw={"width_ratios": [1, 1.6, 1.3]})
plot_points(axes[0], ef_sub.assign(all="all trials"), "all", "comparison", "encoding fidelity",
            hue_order=SETS)
plot_points(axes[1], ef_op.assign(set="target"), OPERATION_COL, "set", "target fidelity by operation (cue comes later)", order=OPS)
plot_points(axes[2], ef_val, VALENCE_COL, "comparison", "encoding fidelity by valence", order=VALENCES,
            hue_order=SETS)
fig.tight_layout()
figures["5_encoding_fidelity"] = fig
plt.show()
display(fidelity_anova)
display(fidelity_pairs)
print(f"target fidelity, {VALENCE_CONTRAST[0]} minus {VALENCE_CONTRAST[1]}:")
display(fidelity_valence)
tables.update(encoding_fidelity_anova=fidelity_anova, encoding_fidelity_pairwise=fidelity_pairs,
              encoding_fidelity_valence=fidelity_valence)

### 5b. Encoding fidelity by scope (Kim et al. 2020, Fig. 5a style)

Target, same subcategory, same category and others for all items and for each category and valence × category, one panel per scope. Dots are subjects (each subject's mean over that scope's trials); the large marker is the mean ± 1 SEM over subjects. Brackets are Tukey-Kramer comparisons with the repeated-measures error term (as in Kim et al. 2020): \*\*\* p < .001, \*\* p < .01, \* p < .05, n.s.

`others` is every template of the other categories. With the default config that includes the neutral objects; with the SPM12-aligned config it is the other of face / place only.

In [ ]:
SCOPES = [("All Items", {})] + [(c.title(), {CATEGORY_COL: c}) for c in ("face", "place")] + [
    (f"{v.title()} {c.title()}", {VALENCE_COL: v, CATEGORY_COL: c}) for c in ("face", "place") for v in VALENCES]
SET_ORDER = SETS
PAIRS = [(SET_ORDER[0], x) for x in SET_ORDER[1:]] + ([(SET_ORDER[-2], SET_ORDER[-1])] if len(SET_ORDER) > 2 else [])
SET_COLORS = {"target": "#C2187A", "related_subcategory": "#A0507F", "related_category": "#C99AB8", "related": "#A0507F",
              "others": "#9A9A9A"}

def stars(p):
    return "n.s." if not np.isfinite(p) or p >= .05 else "***" if p < .001 else "**" if p < .01 else "*"

scope_rows, scope_tests = [], []
fig, axes = plt.subplots(2, 4, figsize=(16, 8.2))
rng = np.random.default_rng(SEED)
for ax, (title, where) in zip(axes.flat, SCOPES):
    part = ef_all
    for col, val in where.items():
        part = part[part[col] == val]
    s = subject_means(part, ["comparison"])
    s = s[s["comparison"].isin(SET_ORDER)]
    tk = rm_tukey(s, "z", "comparison", SET_ORDER, pairs=PAIRS)
    scope_rows.append(s.assign(scope=title))
    scope_tests.append(tk.assign(scope=title))
    for i, comp in enumerate(SET_ORDER):
        v = s[s["comparison"] == comp]["z"].dropna()
        ax.scatter(i + rng.uniform(-0.06, 0.06, len(v)), v, s=11, color=SET_COLORS[comp], alpha=0.35, lw=0)
        ax.errorbar(i, v.mean(), yerr=v.sem(), fmt="o", ms=8, mfc=SET_COLORS[comp], mec="k", ecolor="k",
                    capsize=3, lw=1.2, zorder=4)
    ax.axhline(0, color="0.6", lw=0.8)
    top, span = s["z"].max(), s["z"].max() - s["z"].min()
    for k, (a, b) in enumerate(PAIRS):
        row = tk[(tk["a"] == a) & (tk["b"] == b)]
        p = row["p_tukey"].iloc[0] if len(row) else np.nan
        y = top + span * (0.06 + 0.11 * k)
        x0, x1 = SET_ORDER.index(a), SET_ORDER.index(b)
        ax.plot([x0, x0, x1, x1], [y - span * 0.02, y, y, y - span * 0.02], color="k", lw=0.8)
        ax.text((x0 + x1) / 2, y + span * 0.01, stars(p), ha="center", va="bottom", fontsize=8)
    ax.set_ylim(min(s["z"].min(), 0) - span * 0.08, top + span * (0.12 + 0.11 * len(PAIRS)))
    ax.set_xticks(range(len(SET_ORDER)), [SET_LABELS.get(c, c) for c in SET_ORDER], fontsize=8, rotation=20)
    ax.set_xlim(-0.4, len(SET_ORDER) - 0.6)
    ax.set_title(f"{title} (n = {s['subject'].nunique()})", fontsize=10)
for ax in axes[:, 0]:
    ax.set_ylabel("Fisher z similarity", fontsize=9)
axes.flat[-1].axis("off")
fig.suptitle(f"Figure 5a style localizer-to-task item identification ({ROI}, {LEVEL}-weighted, {FIDELITY_SET})", fontsize=13)
fig.tight_layout()
figures["5b_encoding_fidelity_by_scope"] = fig
plt.show()
encoding_by_scope_tukey = pd.concat(scope_tests, ignore_index=True)
display(encoding_by_scope_tukey)
tables.update(encoding_fidelity_by_scope=pd.concat(scope_rows, ignore_index=True),
              encoding_fidelity_by_scope_tukey=encoding_by_scope_tukey)

## 6. Proactive interference: encoding fidelity of trial N+1

Does the operation on trial N change how well trial N+1's image is encoded when it is from the **same category** (face after face, place after place) rather than a different one? The previous trial is the preceding view in the same run, taken from the full trial sequence. Each run is a single valence, so the comparison is always within valence.

The measure is same minus different category target fidelity, per subject and per operation on trial N; below 0 means proactive interference. Kim 2020 dropped replace trials whose new category matched trial N+1. Clearvale's switch targets are objects, never faces or places, so no trials are dropped.

Tests: a one-sample t of same minus different per operation (with a subject-bootstrap CI), a repeated-measures ANOVA across operations, and FDR-corrected pairwise comparisons.

As in Kim et al. 2020 (Fig. 5b), the planned comparisons (suppress vs. clear, maintain and switch) also get Tukey-Kramer p-values with the repeated-measures error term, and an equal-trial bootstrap (`NPLUS1_BOOTSTRAP_TRIALS` trials drawn per subject × operation × same/different cell, `N_BOOTSTRAP` iterations, 5–95% interval) checks that unequal trial counts don't drive the result.

In [ ]:
prev_cond, same_cat = f"prev_{OPERATION_COL}", f"same_{CATEGORY_COL}"
seq = add_previous_trial_columns(ef_target, [CATEGORY_COL, OPERATION_COL])
seq = seq[seq[same_cat].notna() & seq[prev_cond].isin(OPS)]
print("N+1 trials per previous operation x same/different:")
display(seq.drop_duplicates(["subject", "run_id", "event_index"]).groupby([prev_cond, same_cat]).size().unstack())

pi_sub = subject_means(seq, [prev_cond, same_cat])
interference = paired_contrast(pi_sub, same_cat, "same", "different", by=[prev_cond])
pi_diff = difference_scores(pi_sub, same_cat, "same", "different", by=[prev_cond])
F, p, n = rm_anova(pi_diff, "diff", prev_cond)
interference_anova = pd.DataFrame([{"test": f"rm_anova of same-different across {prev_cond}", "F": F, "p": p, "n": n}])
interference_pairs = pairwise_tests(pi_diff, "diff", prev_cond, levels=[o for o in OPS if o in set(pi_diff[prev_cond])])
interference_tukey = rm_tukey(pi_diff, "diff", prev_cond, OPS, pairs=NPLUS1_PLANNED_PAIRS)
if NPLUS1_BOOTSTRAP_TRIALS:
    boot_summary, boot_pairs = equal_trial_bootstrap(seq, prev_cond, same_cat, n_trials=NPLUS1_BOOTSTRAP_TRIALS)

fig, ax = plt.subplots(figsize=(6, 4))
plot_points(ax, pi_diff.assign(measure="same - different"), prev_cond, "measure",
            "encoding fidelity of trial N+1: same - different category", value="diff",
            order=[o for o in OPS if o in set(pi_diff[prev_cond])])
ax.set_xlabel("operation on trial N")
fig.tight_layout()
figures["6_proactive_interference"] = fig
plt.show()
display(interference)
display(interference_anova)
display(interference_pairs)
print("Planned comparisons (Kim Fig. 5b), Tukey-Kramer with the repeated-measures error term:")
display(interference_tukey)
tables.update(proactive_interference=interference, proactive_interference_anova=interference_anova,
              proactive_interference_pairwise=interference_pairs, proactive_interference_tukey=interference_tukey)
if NPLUS1_BOOTSTRAP_TRIALS:
    print(f"Equal-trial bootstrap ({NPLUS1_BOOTSTRAP_TRIALS} trials per subject x cell, {N_BOOTSTRAP} iterations; 5-95% interval):")
    display(boot_summary)
    display(boot_pairs)
    tables.update(proactive_interference_bootstrap=boot_summary, proactive_interference_bootstrap_pairs=boot_pairs)

## Save outputs

Everything is written to `OUTPUT_DIR`:
- `stats_and_figures.pdf`, every figure above, in order
- one CSV per table
- `run_settings.json`, the settings of this run, including the subjects included

In [ ]:
run_settings = {k: globals()[k] for k in (
    "CONFIG_PATH", "RESULTS_DIR", "ROI", "LEVEL", "TIMECOURSE_SET", "FIDELITY_SET", "OPERATION_COL", "REFERENCE",
    "OPERATIONS", "VALENCE_COL", "VALENCES", "CATEGORY_COL", "SWITCH_SETS", "BASELINE_SECONDS", "STAT_WINDOW_SECONDS",
    "BIN_SECONDS", "PRESET", "BASELINE_MODE", "COMPLETE_TRIALS_ONLY", "VALENCE_CONTRAST", "NPLUS1_PLANNED_PAIRS",
    "NPLUS1_BOOTSTRAP_TRIALS", "ALPHA", "FDR_METHOD", "N_BOOTSTRAP", "SEED", "TR", "BASELINE_TRS", "FIRST_TR", "LAST_TR", "BINS")}
run_settings["subjects"] = sorted(tc["subject"].unique())
run_settings["inclusion_list"] = INCLUDED
for path in save_outputs(OUTPUT_DIR, figures, tables, run_settings):
    print(path)

## Methods

**Scores (core pipeline).** Each image's template is its localizer pattern: z-scored, shifted 4.6 s, and averaged over its 3 presentations. It is restricted to the voxels selective for its category (category vs. others t-contrast, p < .05, ≥ 10-voxel clusters, within the VVS ROI) and weighted by its item-vs-other-items t-map (least-squares-all GLM). Each WM TR is weighted by the same image's t-map and correlated with the template. Pearson r is averaged over the images of a set and Fisher-z transformed. The sets are:
- `target`: the trial's image
- `related_subcategory`: the other images of its subcategory (same valence and category)
- `related_category`: the other images of its category (both valences)
- `others`: the images of the other categories

**Timecourses.** Trials are averaged within subject, with subjects as the random effect. Each curve is baseline-corrected by its mean over the view period, and the removal curves subtract maintain's curve at each TR. The tested window is split into 3-TR bins, each tested with:
- one-sample t-tests against 0, with FDR correction across bins per operation; the removal onset is the first bin with uncorrected p < ALPHA and a mean below 0, as in Kim 2020
- repeated-measures ANOVAs across operations
- FDR-corrected paired t-tests between operations

For valence, removal is computed within valence. It is compared with paired t-tests per bin and with a 3 × 2 repeated-measures ANOVA on the whole-window mean.

**Encoding fidelity and proactive interference.** Encoding fidelity is the window-mean similarity of the view pattern. Proactive interference is same minus different category encoding fidelity of trial N+1, by the operation on trial N. It is tested with one-sample t-tests with subject-bootstrap CIs, a repeated-measures ANOVA across operations, and FDR-corrected pairwise tests.

**Differences from Kim 2020.**
- Clearvale uses four operations (maintain, suppress, switch, clear), with valence as a run-level factor.
- The switch targets are objects rather than a face or scene category.
- bids-hcp preprocessing replaces FSL, and nilearn GLMs replace SPM.
- FDR-corrected paired t-tests replace the Tukey-Kramer post hocs.
- The trial-balanced bootstrap is not reproduced.

See `docs/lab_methods.md` and `examples/clearvale/analysis_deviations.md`.